# 🧬 End-to-End LLM Fine-Tuning
## Non-Instruction → Instruction → Preference Tuning
### Har Line Par Roman Urdu Comments Ke Saath

> **Teen Stages:**
> - **Stage 1** → Raw pharma text se domain-adaptive continued pretraining (Non-Instruction FT)
> - **Stage 2** → Instruction fine-tuning — Q&A format sikhana
> - **Stage 3** → Preference tuning ke liye base ready karna (chosen/rejected data)


---
# 🔵 Stage 1: Non-Instruction Causal LLM Fine-Tuning
## Domain-Adaptive Continued Pretraining

**Maqsad:** Pharma PDF se raw text lekar TinyLlama model ko pharma ki bhasha sikhana.
Model Q&A nahi sikhta — sirf pharma domain ka text pattern sikhta hai.


In [ ]:
# Yeh sirf check karta hai ke notebook theek chal rahi hai
# Agar "all ok" print ho toh sab kuch sahi setup hai
print("all ok")


In [ ]:
from google.colab import userdata
read = userdata.get('token_read_hf')
write = userdata.get('write_token_hf')

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!hf auth whoami

## Pipeline

```text
Pharma PDF
   ↓
PDF text extraction
   ↓
Text cleaning and normalization
   ↓
Data creation
   ↓
Hugging Face Dataset Conversion
   ↓
Tokenization
   ↓
LoRA/QLoRA fine-tuning
   ↓
Validation loss
   ↓
Adapter saving and reloading
   ↓
Text continuation inference
```

## Continued Pretraining vs Instruction Fine-Tuning

In this notebook, we are performing **continued pretraining / non-instruction fine-tuning** on raw pharma PDF text.

The model is given raw domain text such as:

> Metformin is one of the most widely prescribed oral antihyperglycemic agents...

The model then learns to **predict the next token** from this raw text.

This means the model learns:

- Pharma language
- Drug names
- Medical terminology
- Scientific writing style
- Domain-specific sentence patterns

However, the model is **not explicitly taught**:

- How to answer a user's question
- How to follow instructions
- How to respond in Q&A format
- How to behave like a domain-specific chatbot

---

## What Instruction Fine-Tuning Looks Like

In instruction fine-tuning, the training data is prepared in an **instruction-response format**.

Example:

```json
{
  "instruction": "Explain the mechanism of action of Metformin.",
  "input": "",
  "output": "Metformin primarily activates AMPK, which improves glucose uptake and reduces hepatic gluconeogenesis."
}

{
  "messages": [
    {
      "role": "user",
      "content": "What is the primary mechanism of action of Metformin?"
    },
    {
      "role": "assistant",
      "content": "Metformin primarily works by activating AMPK..."
    }
  ]
}

## Pipeline

```text
Non-instrcution FT(RAW)
      ↓
will save the model
      ↓
load the model
      ↓
I will perform instruction FT on same model(question/answer data)
      ↓
will save our model
      ↓
again will load the same model
      ↓
will perform the preference tuning on top of it(choosed/ reject data)
   
```

we are going to train the LORA adapter

## 🔧 Step 1: Zaroori Libraries Install Karna

In [ ]:
# ============================================================
# 1. Libraries Install Karna
# ============================================================
# !pip  → Jupyter ke andar se pip command chalao
# -q    → Quiet mode: zyada output mat dikhao
# -U    → Upgrade: agar pehle se hai toh update karo

# pymupdf      → PDF se text nikalne ke liye (fitz library deta hai)
# datasets     → Hugging Face Dataset format ke liye
# transformers → Model, Tokenizer, Trainer — HuggingFace ka main package
# accelerate   → Training GPU pe fast karne ke liye
# peft         → LoRA/QLoRA adapters — sirf thodi parameters train karo
# bitsandbytes → 4-bit/8-bit quantization — model kam memory mein chalao
# torchao      → PyTorch optimization tools

!pip install -q -U pymupdf datasets transformers accelerate peft bitsandbytes torchao


In [ ]:
# Python warnings system ko configure karo
import warnings

# filterwarnings("ignore") → sari warnings band kar do
# Training ke dauran bohot saari deprecation warnings aati hain — inhe chhupaate hain
warnings.filterwarnings("ignore")


## ⚙️ Step 3: Global Configuration — Saari Settings Ek Jagah

In [ ]:
# ============================================================
# 3. Global Configuration Class
# ============================================================
# dataclass decorator Python class ko automatically:
#   - __init__() method deta hai (manually nahi likhna padta)
#   - __repr__() method deta hai (print karne pe saaf dikhti hai)
# asdict() → dataclass object ko plain dictionary mein convert karta hai

from dataclasses import dataclass, asdict

@dataclass
class Config:

    # ── PDF Settings ─────────────────────────────────────────────────────────
    # Woh PDF file jis se pharma training data nikala jayega
    # Colab mein apni file upload karke yeh path update karo
    pdf_path: str = "/content/Metformin-Lipid-Therapy-Knowledge.pdf"

    # ── Model Settings ────────────────────────────────────────────────────────
    # TinyLlama = 1.1 Billion parameters wala Llama-architecture model
    # Isliye choose kiya: lightweight, free GPU pe chalata hai, beginner-friendly
    model_name: str = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

    # ── Output Directories ────────────────────────────────────────────────────
    # output_dir       → Training ke dauran checkpoints yahan save honge
    output_dir: str = "/content/pharma_tinyllama_lora_output"

    # adapter_dir      → Final trained LoRA adapter ki files yahan jayengi
    adapter_dir: str = "/content/pharma_tinyllama_lora_adapter"

    # processed_data_dir → Cleaned paragraphs aur raw pages JSONL files
    processed_data_dir: str = "/content/pharma_processed_data"

    # ── Text Preprocessing ────────────────────────────────────────────────────
    # min_chars_per_paragraph = 80
    # 80 characters se chote paragraphs training mein shamil nahi honge
    # Woh headings, page numbers ya noise hote hain — koi kaam ke nahi
    min_chars_per_paragraph: int = 80

    # block_size = 512
    # Ek training example mein exactly 512 tokens honge
    # Yeh sequence length hai — embedding size NAHI hai
    # Matlab: model ek baar mein 512 tokens dekhega
    block_size: int = 512

    # ── Dataset Split ─────────────────────────────────────────────────────────
    # test_size = 0.15 → 15% data validation ke liye, 85% training ke liye
    test_size: float = 0.15

    # seed = 42 → Reproducible random split — har baar same data train/val mein
    seed: int = 42

    # ── LoRA Parameters ───────────────────────────────────────────────────────
    # lora_r = 16  (Rank)
    # Adapter matrix ki size: chota r = kam parameters = fast lekin less accurate
    # r=16: ek 4096x4096 layer ki jagah sirf 4096x16 + 16x4096 train hogi
    # Parameters: 4096*16*2 = 131K vs original 16.7M — 127x compression!
    lora_r: int = 16

    # lora_alpha = 32  (Scaling Factor)
    # LoRA update ka strength = alpha / r = 32 / 16 = 2.0
    # Zyada alpha = adapter ka zyada asar — aggressive fine-tuning
    lora_alpha: int = 32

    # lora_dropout = 0.05  (Regularization)
    # Training mein 5% LoRA neurons randomly off karo — overfitting rokta hai
    lora_dropout: float = 0.05

    # ── Training Hyperparameters ──────────────────────────────────────────────
    # num_train_epochs = 10.0
    # Pura dataset 10 baar model ko dikhao
    # Ek epoch = ek baar poora training data dekha
    num_train_epochs: float = 2.0

    # per_device_train_batch_size = 1
    # Ek step mein sirf 1 training example — small GPU ke liye safe
    per_device_train_batch_size: int = 1

    # per_device_eval_batch_size = 1
    # Validation ke dauran bhi ek baar mein 1 example
    per_device_eval_batch_size: int = 1

    # gradient_accumulation_steps = 8
    # 8 micro-steps ke baad ek optimizer update karo
    # Effective batch size = 1 * 8 = 8 — bina extra memory ke bada batch simulate
    gradient_accumulation_steps: int = 8

    # learning_rate = 2e-4 (0.0002)
    # Optimizer ka step size — zyada = fast lekin unstable, kam = slow lekin stable
    # LoRA ke liye 1e-4 to 3e-4 best range hai
    learning_rate: float = 2e-4

    # warmup_ratio = 0.03
    # Pehle 3% training steps mein learning rate 0 se target tak slowly barhao
    warmup_ratio: float = 0.03

    # weight_decay = 0.01
    # L2 regularization — bade weights penalize karo, overfitting roko
    weight_decay: float = 0.01

    # logging_steps = 1 → Har ek step ke baad training loss print karo
    # logging_first_step = True → Pehle step ka log zaroor print karo
    logging_steps=1
    logging_first_step=True

    # eval_steps = 10 → Har 10 steps pe validation set pe evaluate karo
    eval_steps: int = 10

    # save_steps = 25 → Har 25 steps pe checkpoint save karo
    save_steps: int = 25

    # save_total_limit = 2 → Sirf 2 checkpoints rakhho, purane delete karo
    save_total_limit: int = 2

    # max_steps = -1 → -1 matlab epochs se control karo (puri training)
    # Quick demo ke liye ise 20 ya 30 kar sakte hain
    max_steps: int = -1


In [ ]:
# Config class ka object banao — sari default values load ho jaati hain
# Poore notebook mein config.model_name, config.lora_r waghera use honge
config = Config()


In [ ]:
# Config object display karo — Jupyter mein last line automatically print hoti hai
config


In [ ]:
# Config ki saari values ko JSON format mein print karo
# asdict() → dataclass → dictionary
# json.dumps() → dictionary → formatted string
# indent=2   → 2 spaces indentation — human-readable
import json
print(json.dumps(asdict(config), indent=2))


In [ ]:
# Sirf output directory ka path check karo
# Yahan training ke dauran model checkpoints save honge
config.output_dir


In [ ]:
# Processed data folder ka path confirm karo
# Yahan cleaned paragraphs aur raw PDF pages save honge
config.processed_data_dir


In [ ]:
# Teen zaroori folders banao agar pehle se exist na karein
# os.makedirs() → folder banao (nested folders bhi)
# exist_ok=True → agar folder pehle se hai toh error mat do
import os
os.makedirs(config.output_dir, exist_ok=True)         # Training output
os.makedirs(config.adapter_dir, exist_ok=True)        # LoRA adapter save
os.makedirs(config.processed_data_dir, exist_ok=True) # Cleaned data save


In [ ]:
# ============================================================
# Hugging Face repo names
# ============================================================

HF_USERNAME = "iabubakar"

BASE_MODEL_NAME = config.model_name

# Stage 1: Non-instruction LoRA adapter
HF_REPO_NON_INSTRUCTION_ADAPTER = f"{HF_USERNAME}/pharma-tinyllama-non-instruction-lora-adapter"

# Stage 1 merged model
HF_REPO_NON_INSTRUCTION_MERGED = f"{HF_USERNAME}/pharma-tinyllama-non-instruction-merged"

# Stage 2: Instruction LoRA adapter
HF_REPO_INSTRUCTION_ADAPTER = f"{HF_USERNAME}/pharma-tinyllama-instruction-lora-adapter"

# Stage 2 merged model
HF_REPO_INSTRUCTION_MERGED = f"{HF_USERNAME}/pharma-tinyllama-instruction-merged"

# Stage 3: DPO preference LoRA adapter
HF_REPO_DPO_ADAPTER = f"{HF_USERNAME}/pharma-tinyllama-dpo-lora-adapter"

# Stage 3 final merged model
HF_REPO_DPO_MERGED = f"{HF_USERNAME}/pharma-tinyllama-dpo-merged"

print(HF_REPO_NON_INSTRUCTION_ADAPTER)
print(HF_REPO_INSTRUCTION_ADAPTER)
print(HF_REPO_DPO_ADAPTER)

## 📄 Step 4: PDF File Ka Wujood Check Karna

In [ ]:
# ============================================================
# 4. PDF File Check Karo
# ============================================================
# os.path.exists() → File/folder hai ya nahi — True/False
# Agar PDF nahi mili toh user ko warning do taake woh upload kare

if not os.path.exists(config.pdf_path):
    # PDF nahi mili — user ko batao
    print(f"PDF not found at: {config.pdf_path}")
else:
    # PDF hai — aage badhao
    print(f"PDF found: {config.pdf_path}")


## 📖 Step 5: PDF Se Text Nikalna

In [ ]:
# ============================================================
# 5. PDF Se Text Extract Karne Ka Function
# ============================================================
# List[Dict[str, Any]] → Return type: dictionaries ki list
# Har dictionary mein: page number, text, character count

from typing import List, Dict, Any
import fitz  # PyMuPDF — PDF padhne ki library

def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:

    pages = []  # Yahan har page ki info store hogi

    # fitz.open() → PDF file kholta hai
    # 'with' statement → block khatam hone pe file automatically close
    with fitz.open(pdf_path) as doc:

        # enumerate(doc, start=1) → har page pe loop, page_index 1 se shuru
        for page_index, page in enumerate(doc, start=1):

            # page.get_text("text") → is page ka plain text nikalo
            # "text" mode → sirf text, koi layout ya formatting nahi
            text = page.get_text("text")

            # .strip() → aage peeche ke whitespace hatao
            # Agar text None ho toh empty string use karo
            text = text.strip() if text else ""

            # Sirf woh pages rakho jinme kuch text ho
            if text:
                pages.append({
                    "page": page_index,       # Page number (1 se start)
                    "text": text,             # Page ka extracted text
                    "char_count": len(text),  # Kitne characters hain
                })

    return pages  # Pages ki list wapas karo


In [ ]:
# Config mein set kiya gaya PDF path confirm karo
config.pdf_path


In [ ]:
# Function call karo — PDF se sab pages ka text nikalo
# Result: list of dicts, har dict mein page ki info
pdf_pages = extract_pdf_pages(config.pdf_path)


In [ ]:
# Total pages aur har page ka character count print karo
print(f"Total pages with extracted text: {len(pdf_pages)}")
print("Page-level character counts:")
for item in pdf_pages:
    print(f"Page {item['page']}: {item['char_count']} characters")


In [ ]:
# Pehle page ka raw text dekho — verify karo extraction sahi hua
# pdf_pages[0] → pehli dictionary, ["text"] → us ki text key
print(pdf_pages[0]["text"])


## 🧹 Step 6: Text Cleaning Utilities

PDF se nikla text gandha hota hai — hidden characters, toote hue words, extra spaces.
Yeh step sab clean karta hai taake model ko sahi data mile.


| Cleaning Step                          | Code / Logic                             | What It Does                                                                  | Example Before                                                  | Example After                                                  | Why It Matters for Fine-Tuning                                            |
| -------------------------------------- | ---------------------------------------- | ----------------------------------------------------------------------------- | --------------------------------------------------------------- | -------------------------------------------------------------- | ------------------------------------------------------------------------- |
| Unicode normalization                  | `unicodedata.normalize("NFKC", text)`    | Converts unusual Unicode characters into standard readable characters.        | `ＡＭＰＫ`, `ﬁ`                                                     | `AMPK`, `fi`                                                   | Prevents tokenizer confusion caused by hidden or non-standard characters. |
| Remove zero-width characters           | `text.replace("\u200b", "")`             | Removes invisible zero-width spaces from PDF text.                            | `Metformin​ activates AMPK`                                     | `Metformin activates AMPK`                                     | Invisible characters can create bad tokens and noisy training data.       |
| Remove BOM / hidden marker             | `text.replace("\ufeff", "")`             | Removes hidden Byte Order Mark characters sometimes found in extracted text.  | `﻿Metformin is used...`                                         | `Metformin is used...`                                         | Keeps the training text clean and consistent.                             |
| Fix hyphenated line breaks             | `re.sub(r"(\w)-\n(\w)", r"\1\2", text)`  | Joins words that were broken across PDF lines.                                | `gluconeogene-\nsis`                                            | `gluconeogenesis`                                              | Prevents the model from learning broken medical terms.                    |
| Normalize spaces and tabs              | `re.sub(r"[ \t]+", " ", text)`           | Converts multiple spaces or tabs into one space.                              | `Metformin     activates    AMPK`                               | `Metformin activates AMPK`                                     | Makes text consistent and easier for tokenizer/model to learn.            |
| Normalize blank lines                  | `re.sub(r"\n{3,}", "\n\n", text)`        | Converts too many blank lines into a proper paragraph gap.                    | `Para 1\n\n\n\nPara 2`                                          | `Para 1\n\nPara 2`                                             | Preserves paragraph structure without unnecessary whitespace noise.       |
| Remove standalone page numbers         | `re.sub(r"(?m)^\s*\d+\s*$", "", text)`   | Removes lines that contain only page numbers.                                 | `1` or `23`                                                     | Removed                                                        | Prevents the model from learning irrelevant PDF page numbers.             |
| Split into paragraphs                  | `re.split(r"\n\s*\n", text)`             | Splits text wherever there is a blank line.                                   | `Para 1\n\nPara 2`                                              | `["Para 1", "Para 2"]`                                         | Helps preserve meaningful document structure.                             |
| Remove line wrapping inside paragraphs | `re.sub(r"\n+", " ", paragraph)`         | Converts broken lines inside the same paragraph into a single paragraph line. | `Metformin is widely prescribed\noral antihyperglycemic agent.` | `Metformin is widely prescribed oral antihyperglycemic agent.` | Prevents the model from learning artificial PDF line breaks.              |
| Normalize paragraph spacing            | `re.sub(r"\s+", " ", paragraph).strip()` | Removes extra spaces inside each paragraph and trims start/end spaces.        | `  Metformin   activates   AMPK.  `                             | `Metformin activates AMPK.`                                    | Produces clean, readable training examples.                               |
| Remove empty paragraphs                | `if paragraph:`                          | Keeps only non-empty cleaned paragraphs.                                      | `""`                                                            | Removed                                                        | Avoids useless blank samples in the dataset.                              |
| Rebuild cleaned text                   | `"\n\n".join(cleaned_paragraphs)`        | Joins cleaned paragraphs with two newlines.                                   | List of cleaned paragraphs                                      | Clean paragraph-level text                                     | Creates a clean corpus suitable for causal LM training.                   |
| Track cleaned page length              | `char_count: len(cleaned_text)`          | Stores number of characters after cleaning.                                   | Raw page length unknown                                         | `char_count = 1450`                                            | Helps debug whether a page has too little or too much extracted content.  |
| Preview cleaned output                 | `cleaned_pages[0]["text"][:1500]`        | Prints first 1500 characters of cleaned page 1.                               | Full cleaned page                                               | Preview text                                                   | Helps manually verify that cleaning worked correctly.                     |


In [ ]:
# ============================================================
# 6. Text Cleaning Utilities
# ============================================================


In [ ]:
import re
import unicodedata

def clean_pdf_text(text: str) -> str:
    # Standardize Unicode text so visually similar characters are treated consistently.
    # Example: "ＡＭＰＫ" becomes "AMPK" and "ﬁ" becomes "fi".
    text = unicodedata.normalize("NFKC", text)

    # Remove invisible characters that may appear during PDF text extraction.
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # Join words broken by line hyphenation, e.g., "gluconeogene-\nsis" -> "gluconeogenesis".
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Replace multiple spaces/tabs with a single space.
    text = re.sub(r"[ \t]+", " ", text)

    # Convert three or more newlines into a standard paragraph break.
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove lines that contain only page numbers.
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Split text into paragraphs, clean each paragraph, and remove empty ones.
    paragraphs = []
    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = re.sub(r"\n+", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()

        if paragraph:
            paragraphs.append(paragraph)

    # Join cleaned paragraphs with one blank line between them.
    return "\n\n".join(paragraphs)

In [ ]:
# Cleaned pages ke liye khali list banao
cleaned_pages = []


In [ ]:
# Har raw page pe clean_pdf_text() function chalao
for page in pdf_pages:
    # Is page ka text clean karo
    cleaned_text = clean_pdf_text(page["text"])

    # Result list mein add karo
    cleaned_pages.append({
        "page": page["page"],           # Page number same rakho
        "text": cleaned_text,           # Cleaned text
        "char_count": len(cleaned_text),# Cleaned text ki length
    })


In [ ]:
# Total cleaned pages ki ginti print karo
print("Total cleaned pages:", len(cleaned_pages))


In [ ]:
# Pehle cleaned page ka text dekho — verify karo cleaning sahi hui
print(cleaned_pages[0]["text"])


## ✂️ Step 7: Paragraphs Mein Batna

In [ ]:
# ============================================================
# 7. Cleaned Pages Ko Paragraph Records Mein Convert Karna
# ============================================================
# Har page ke text ko individual paragraphs mein toRte hain
# Yeh training data ki basic units hain

# cleaned_pages → clean pages ki list
# min_chars=80  → 80 se chote paragraphs ignore karo (woh noise hain)
def split_into_paragraph_records(cleaned_pages, min_chars=80):
    paragraph_records = []  # Final records yahan store honge

    for page in cleaned_pages:

        # "\n\n" pe split karo → blank line paragraph boundary hai
        paragraphs = page["text"].split("\n\n")

        # enumerate(paragraphs, start=1) → index 1 se shuru
        for paragraph_index, paragraph in enumerate(paragraphs, start=1):

            # Paragraph ke aage peeche se whitespace hatao
            paragraph = paragraph.strip()

            # Chote paragraphs skip karo
            # Woh usually: headings, "Table 1", page numbers, ya sirf noise hain
            if len(paragraph) < min_chars:
                continue

            # Valid paragraph — record mein save karo
            paragraph_records.append({
                "text": paragraph,               # Paragraph ka text
                "source_page": page["page"],     # Kis page se aaya
                "paragraph_id": paragraph_index, # Is page pe kaun sa paragraph
                "char_count": len(paragraph),    # Kitne characters hain
            })

    return paragraph_records


In [ ]:
# Function call karo — default min_chars=80 use hogi
paragraph_records = split_into_paragraph_records(cleaned_pages)


In [ ]:
# Total paragraphs ki ginti print karo
print("Total paragraph records:", len(paragraph_records))


In [ ]:
# Pehle 3 paragraph records preview karo — verify karo data sahi hai
for record in paragraph_records[:3]:
    print("=" * 80)
    print(f"Page: {record['source_page']} | Paragraph: {record['paragraph_id']} | Characters: {record['char_count']}")
    print(record["text"])


## 💾 Step 8: Cleaned Data Save Karna

In [ ]:
# ============================================================
# 8. Intermediate Data Save Karna — Reproducibility Ke Liye
# ============================================================
# Real projects mein hamesha intermediate data save karo kyunki:
# 1. Reproducibility → same data dobara use kar sako
# 2. Debugging      → training fail ho toh data inspect kar sako
# 3. Compliance     → pharma projects mein audit trail zaroori hoti hai
# JSONL format: har line ek JSON object — large datasets ke liye best

# File paths banao
raw_pages_path = os.path.join(config.processed_data_dir, "pdf_pages_raw.jsonl")
paragraphs_path = os.path.join(config.processed_data_dir, "pharma_paragraph_process.jsonl")

# Raw PDF pages save karo
# open(..., "w", encoding="utf-8") → write mode, UTF-8 encoding
with open(raw_pages_path, "w", encoding="utf-8") as f:
    for item in pdf_pages:
        # json.dumps() → dict → JSON string
        # ensure_ascii=False → non-English chars as-is rakhta hai
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

# Cleaned paragraphs save karo
with open(paragraphs_path, "w", encoding="utf-8") as f:
    for item in paragraph_records:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved raw pages to: {raw_pages_path}")
print(f"Saved cleaned paragraph corpus to: {paragraphs_path}")


## 🗂️ Step 9: Hugging Face Dataset Banana

In [ ]:
# ============================================================
# 9. Hugging Face Dataset Object Banana
# ============================================================
# Paragraph records ki list ko training ke liye standardized format mein convert karo

from datasets import Dataset

# Safety check: agar sirf 1 ya 0 paragraphs hain toh split nahi ho sakta
if len(paragraph_records) < 2:
    raise ValueError(
        "The extracted corpus is too small. Please provide a larger pharma PDF or lower min_chars_per_paragraph."
    )

# Dataset.from_list() → Python list → Hugging Face Dataset
# Is format mein .map(), .train_test_split(), .shuffle() easily use ho sakti hain
text_dataset = Dataset.from_list(paragraph_records)


In [ ]:
# Dataset ka overview print karo — columns, rows, data types
print(text_dataset)


In [ ]:
# Pehli entry dekho — verify karo ke data correct format mein hai
print(text_dataset[0])


## 🔀 Step 10: Train/Validation Split

In [ ]:
# ============================================================
# 10. Data Ko Training aur Validation Mein Batna
# ============================================================
# Hamesha validation set rakho — chahe dataset chota hi kyun na ho
# Validation loss se pata chalta hai: model seedh raha hai ya sirf yaad kar raha hai

# train_test_split()
# test_size=0.15 → 15% validation, 85% training
# seed=42        → Reproducible split — har baar same data train/val mein jaye
split_dataset = text_dataset.train_test_split(test_size=config.test_size, seed=config.seed)

from datasets import DatasetDict

# DatasetDict banao — "train" aur "validation" keys ke saath
# split_dataset["test"] ko "validation" naam dete hain
dataset = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"],  # HuggingFace "test" return karta hai
})

print(dataset)  # Dono splits ka size dikhega


## 🔤 Step 11: Tokenizer Load Karna

In [ ]:
# ============================================================
# 11. Tokenizer Load Karna
# ============================================================
# Tokenizer text ko numbers (token IDs) mein badalta hai
# Model sirf numbers samajhta hai, text nahi

from transformers import AutoTokenizer

# AutoTokenizer.from_pretrained() → Hugging Face Hub se tokenizer download/load
# config.model_name → TinyLlama ka tokenizer
# use_fast=True     → Rust-based fast tokenizer (slow Python tokenizer se zyada fast)
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

# TinyLlama aur zyada Llama models mein pad_token defined nahi hota
# Training mein padding ki zarurat hoti hai jab batch mein mixed lengths hon
if tokenizer.pad_token is None:
    # EOS (End-of-Sequence) token ko pad token bana do
    # Causal LM ke liye yeh common aur safe practice hai
    tokenizer.pad_token = tokenizer.eos_token

# padding_side = "right" → padding dayen taraf lagao
# Causal (left-to-right) models ke liye right padding standard hai
tokenizer.padding_side = "right"


In [ ]:
# EOS (End of Sequence) token kya hai — dekho
# Yeh woh special token hai jo sentence/sequence ke khatme pe lagta hai
tokenizer.eos_token


In [ ]:
# Tokenizer ki summary print karo — sab important values
print(f"Tokenizer loaded: {config.model_name}")
print(f"Vocab size: {len(tokenizer)}")   # Kitne unique tokens vocabulary mein
print(f"Pad token: {tokenizer.pad_token} | Pad token id: {tokenizer.pad_token_id}")
print(f"EOS token: {tokenizer.eos_token} | EOS token id: {tokenizer.eos_token_id}")


## 📦 Step 12: Tokenization aur Text Packing

In [ ]:
# ============================================================
# 12. Tokenization Function
# ============================================================
# examples → ek batch of texts (dataset.map() automatically deta hai)
# Return   → token IDs aur attention masks

def tokenize_function(examples):
    # tokenizer(examples["text"]) → text list ko token IDs mein convert karo
    # Padding nahi karte yahan — text packing function baad mein handle karega
    # Truncation bhi nahi — create_training_blocks deal karega
    return tokenizer(examples["text"])


In [ ]:
# Tokenization poori dataset pe apply karo
# dataset.map() → har example pe function chalao
# remove_columns=... → original "text", "source_page" etc. columns hata do
#                      (sirf token IDs chahiye ab)
# desc="..." → progress bar ka message
tokenized_datasets = dataset.map(
    tokenize_function,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing text corpus",
)


In [ ]:
# Tokenized dataset ka structure dekho
# Abhi "input_ids" aur "attention_mask" columns hain
# Har row mein alag alag length ke tokens hain
tokenized_datasets


In [ ]:
# Training set ka pehla example ke token IDs dekho
# Yeh numbers hain jo tokenizer ne text ke liye assign kiye
tokenized_datasets['train']['input_ids'][0]


## 🗜️ Text Packing — Efficient Training Blocks Banana

**Bina packing ke (Approach 1):**
Har paragraph separately 512 tokens tak pad/truncate ho jata hai → bohot padding waste.

**Text Packing ke saath (Approach 2 — is notebook mein):**
Sab tokens ek lambi stream mein joRo → 512-token blocks mein kato → zero padding waste.


In [ ]:
# ============================================================
# create_training_blocks() — Text Packing Function
# ============================================================
# Yeh function tokenized texts ko efficient 512-token blocks mein convert karta hai
# Sab tokens pehle ek lambi list mein joRe jaate hain, phir fixed size mein kaate hain

def create_training_blocks(tokenized_examples):

    # ── Step 1: Sab Token IDs Ek List Mein Jodo ──────────────────────────────
    # tokenized_examples["input_ids"] → [[101, 234, ...], [45, 67, ...], ...]
    # .extend() → nested list ko flat list mein jodo
    all_input_ids = []
    all_attention_masks = []

    for input_ids in tokenized_examples["input_ids"]:
        all_input_ids.extend(input_ids)       # Har example ke tokens joRo

    for attention_mask in tokenized_examples["attention_mask"]:
        all_attention_masks.extend(attention_mask)  # Har example ka mask joRo

    # ── Step 2: Kitne Complete Blocks Ban Sakte Hain ─────────────────────────
    total_tokens = len(all_input_ids)
    # (total // 512) * 512 → 512 ka closest chota multiple
    # Example: 1300 → (1300 // 512) * 512 = 2 * 512 = 1024 (276 tokens drop honge)
    usable_tokens = (total_tokens // config.block_size) * config.block_size

    # Agar ek bhi poora block nahi ban sakta toh khali data return karo
    if usable_tokens == 0:
        return {"input_ids": [], "attention_mask": [], "labels": []}

    # ── Step 3: Remainder Tokens Drop Karo ───────────────────────────────────
    # Sirf itne tokens rakho jinse poore blocks ban sakein
    all_input_ids = all_input_ids[:usable_tokens]
    all_attention_masks = all_attention_masks[:usable_tokens]

    # ── Step 4: 512-Token Blocks Banao ───────────────────────────────────────
    input_id_blocks = []
    attention_mask_blocks = []

    # range(0, 1024, 512) → [0, 512] → do iterations
    for start_index in range(0, usable_tokens, config.block_size):
        end_index = start_index + config.block_size

        # Slice: all_input_ids[0:512] = Block 1, all_input_ids[512:1024] = Block 2
        input_id_blocks.append(all_input_ids[start_index:end_index])
        attention_mask_blocks.append(all_attention_masks[start_index:end_index])

    # ── Step 5: Labels = Input IDs (Causal LM) ───────────────────────────────
    # Causal LM mein model input dekhta hai aur next token predict karta hai
    # Labels = input_ids hi hote hain — model internally shift karta hai
    # Token[0] se Token[1] predict karo, Token[1] se Token[2], etc.
    labels = input_id_blocks.copy()

    return {
        "input_ids": input_id_blocks,
        "attention_mask": attention_mask_blocks,
        "labels": labels,
    }


In [ ]:
# Text packing apply karo — tokenized data → 512-token blocks
# batched=True → sab tokens ek saath process karo (packing ke liye zaroori)
final_dataset = tokenized_datasets.map(
    create_training_blocks,
    batched=True,
    desc=f"Creating fixed-size training blocks of {config.block_size} tokens",
)


In [ ]:
# Ek packed example inspect karo — verify karo sab kuch sahi hai
sample = final_dataset["train"][0]


In [ ]:
# Sample ke keys aur lengths check karo
print("Keys:", sample.keys())
print("input_ids length:", len(sample["input_ids"]))   # 512 hona chahiye
print("labels length:", len(sample["labels"]))          # 512 hona chahiye
print("Decoded sample preview:\n")
# Token IDs → text — dekho kya pharma content hai
print(tokenizer.decode(sample["input_ids"][:250]))


## 🤖 Step 13: Base Model Load Karna (QLoRA 4-bit)

GPU available ho toh 4-bit quantized model load karo — bohot kam memory mein chal jata hai.
CPU pe bhi kaam karta hai magar training bohot slow hogi.


In [ ]:
# ============================================================
# 13. GPU Check aur Memory Clear
# ============================================================

import torch

# torch.cuda.is_available() → GPU (CUDA) hai ya nahi — True/False
use_cuda = torch.cuda.is_available()
print("CUDA available:", use_cuda)

if use_cuda:
    # GPU ka naam print karo jaise "Tesla T4", "A100", "RTX 3090"
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Model load karne se pehle memory free karo
import gc

# gc.collect() → Python garbage collector → unused Python objects memory free karo
gc.collect()

if use_cuda:
    # GPU ka cached (reserved but unused) memory bhi free karo
    torch.cuda.empty_cache()


In [ ]:
# ============================================================
# Base Model Load Karna — GPU ya CPU
# ============================================================

from transformers import AutoModelForCausalLM

if use_cuda:
    from transformers import BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training

    # ── 4-bit Quantization Config (QLoRA) ─────────────────────────────────────
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,              # Model weights 4-bit mein store karo
                                        # Normal float32 = 4 bytes per weight
                                        # 4-bit = 0.5 bytes per weight → 8x kam memory!
        bnb_4bit_quant_type="nf4",      # NormalFloat4 — neural networks ke liye best 4-bit format
                                        # Standard int4 se zyada accurate hai
        bnb_4bit_compute_dtype=torch.float16,  # Calculations float16 mein karo
                                               # float32 se 2x fast, accuracy thodi kam
        bnb_4bit_use_double_quant=True, # Double quantization:
                                        # Quantization constants bhi compress karo
                                        # Thodi aur memory bachti hai
    )

    # ── Model GPU Pe Load Karo ────────────────────────────────────────────────
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,                     # TinyLlama model HuggingFace Hub se
        quantization_config=quantization_config, # Upar wali 4-bit settings
        device_map="auto",                     # GPU/CPU automatically decide karo
                                               # Multi-GPU environments pe bhi kaam karta hai
        trust_remote_code=True,                # Model ke custom Python code ko allow karo
    )

    # 4-bit quantized model ko training ke liye prepare karo
    # Yeh function:
    # - Gradient checkpointing enable karta hai
    # - Layer norms float32 mein rakkhta hai
    # - Bina is ke 4-bit training unstable hoti hai
    base_model = prepare_model_for_kbit_training(base_model)

else:
    # ── CPU Mode ─────────────────────────────────────────────────────────────
    # GPU nahi hai toh float32 mein load karo — yeh slow hoga
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,   # Full precision — CPU ke liye zaroori
        trust_remote_code=True,
    )

# Training ke dauran cache disable karo
# use_cache=True → inference speed ke liye — training mein memory waste karta hai
base_model.config.use_cache = False

print("Base model loaded successfully.")


## 🔌 Step 14: LoRA Adapters Lagana

In [ ]:
# ============================================================
# 14. LoRA Adapters Apply Karna
# ============================================================
# LoRA (Low-Rank Adaptation) ka idea:
#   Base model weights → freeze (change mat karo)
#   Har target layer mein 2 choti matrices A aur B inject karo
#   Sirf A aur B train karo (bohot kam parameters)
#   Final output = W_original + (lora_alpha/r) * (A × B)

from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,   # Hum Causal Language Model train kar rahe hain
                                    # CAUSAL_LM = GPT/Llama style (left-to-right prediction)

    r=config.lora_r,                # Rank = 16
                                    # A matrix: (hidden_dim × r), B matrix: (r × hidden_dim)
                                    # Parameters saved: hidden*hidden vs 2*hidden*16
                                    # Example: 4096×4096 = 16.7M vs 2×4096×16 = 131K

    lora_alpha=config.lora_alpha,   # Alpha = 32 (scaling factor)
                                    # LoRA output = (alpha/r) × (A×B) = (32/16) × (A×B) = 2×(A×B)
                                    # Zyada alpha = zyada LoRA influence on final output

    lora_dropout=config.lora_dropout, # 0.05 → training mein 5% neurons randomly off
                                      # Overfitting rokta hai — model specific patterns yaad na kare

    bias="none",                    # Bias parameters train nahi karo
                                    # "none" = sabse memory efficient option
                                    # Alternatives: "all", "lora_only"

    target_modules=[                # In specific attention/FFN layers mein adapters lagao
        "q_proj",   # Query projection  → "Mujhe kya dhundhna hai?"
        "k_proj",   # Key projection    → "Main kisse match karta hoon?"
        "v_proj",   # Value projection  → "Match hone pe kya return karun?"
        "o_proj",   # Output projection → Attention ka final output
        "gate_proj",# FFN gate layer    → SwiGLU activation gate (Llama architecture)
        "up_proj",  # FFN up-projection → Hidden dimension barhao
        "down_proj",# FFN down-proj     → Hidden dimension wapas ghataao
    ],
    # Yeh 7 layers Transformer ki sabse important computation layers hain
    # Inhe train karna = model ko effectively fine-tune karna
)


In [ ]:
# Base model pe LoRA config apply karo
# get_peft_model() → base model mein LoRA adapter layers inject karta hai
# Ab "model" = frozen base_model + trainable LoRA adapters
from peft import get_peft_model
model = get_peft_model(base_model, lora_config)


In [ ]:
# Trainable parameters ki ginti print karo
# Expected output: "trainable params: X || all params: 1.1B || trainable%: ~0.7%"
# Sirf 0.7% parameters train ho rahe hain — baaki sab frozen hain!
model.print_trainable_parameters()


## 🗃️ Step 15: Data Collator

In [ ]:
# ============================================================
# 15. Data Collator — Training Batches Banana
# ============================================================
# Data collator ka kaam: dataset examples → proper training batches
# Trainer aur Dataset ke beech ka bridge hai

from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,  # Tokenizer chahiye padding handle karne ke liye

    mlm=False             # mlm = Masked Language Modeling (BERT ka style)
                          # mlm=True  → kuch tokens mask karo, model predict kare [MASK]
                          # mlm=False → Causal LM: next token predict karo (left to right)
                          # Hum TinyLlama use kar rahe hain → mlm=False zaroori hai
)
# Data collator yeh karta hai:
# Example 1 (512 tokens) + Example 2 (512 tokens) →
# input_ids tensor [2, 512], attention_mask [2, 512], labels [2, 512]


## 🏋️ Step 16: Training Arguments

In [ ]:
# ============================================================
# 16. Training Arguments — Trainer Ko Instructions Dena
# ============================================================

from transformers import TrainingArguments


In [ ]:
# Training settings ki dictionary banao
training_kwargs = dict(
    output_dir=config.output_dir,              # Checkpoints yahan save honge

    num_train_epochs=config.num_train_epochs,  # 10 epochs — pura data 10 baar dikhao

    max_steps=config.max_steps,                # -1 → epochs se control karo
                                               # Positive number → sirf itne steps

    per_device_train_batch_size=config.per_device_train_batch_size,  # 1 example per step
    per_device_eval_batch_size=config.per_device_eval_batch_size,    # Validation mein bhi 1

    gradient_accumulation_steps=config.gradient_accumulation_steps,  # 8 steps → phir update
    # Effective batch size = 1 × 8 = 8
    # Bina extra GPU memory ke bada batch simulate karta hai

    learning_rate=config.learning_rate,   # 0.0002 — optimizer ka step size
                                          # LoRA ke liye 1e-4 to 3e-4 best range

    warmup_steps=5,   # Pehle 5 steps mein LR 0 se 0.0002 tak slowly barhao
                      # Training ke shuru mein stability deta hai

    weight_decay=config.weight_decay,  # 0.01 → L2 regularization — overfitting roko

    # Har ek step ke baad loss print karo (small dataset ke liye useful)
    logging_steps=1,
    logging_first_step=True,   # Pehle step ka log zaroor print karo

    eval_steps=config.eval_steps,         # Har 10 steps pe validation chalao
    save_steps=config.save_steps,         # Har 25 steps pe checkpoint save karo
    save_total_limit=config.save_total_limit, # Sirf 2 checkpoints rakho

    fp16=use_cuda,   # GPU ho toh float16 use karo — 2x fast, half memory
    bf16=False,      # bfloat16 nahi — sirf A100/H100 pe better hota hai

    report_to="none",             # Wandb/TensorBoard ko report nahi karna
    remove_unused_columns=False,  # Extra columns accidentally remove mat karo
)


In [ ]:
# TrainingArguments object banao
# ** (double star) → dictionary ko keyword arguments mein unpack karo
from transformers import TrainingArguments
training_args = TrainingArguments(**training_kwargs)


## 🏗️ Step 17: Trainer Build Karna

In [ ]:
# ============================================================
# 17. Trainer Build Karna
# ============================================================
# Hugging Face Trainer sab kuch manage karta hai:
# - Training loop (forward pass, loss, backward pass, optimizer)
# - Evaluation
# - Checkpoint saving
# - Logging

from transformers import Trainer

trainer = Trainer(
    model=model,                              # Fine-tune hone wala LoRA model
    args=training_args,                       # Upar set ki gayi settings
    train_dataset=final_dataset["train"],     # 512-token training blocks
    eval_dataset=final_dataset["validation"], # Validation blocks
    data_collator=data_collator,              # Batches prepare karne wala
)
print("Trainer is ready.")


In [ ]:
# Warnings ignore karo — training ke dauran deprecation warnings aati hain
import warnings
warnings.filterwarnings("ignore")


## 🚀 Step 18: Training Shuru!

In [ ]:
# ============================================================
# 18. Training Shuru Karo!
# ============================================================
# trainer.train() → poori training pipeline chalti hai:
# Har step mein:
#   1. Batch lo (data_collator se)
#   2. Forward pass → model prediction
#   3. Loss calculate karo (cross-entropy: prediction vs actual next token)
#   4. Backward pass → gradients
#   5. gradient_accumulation_steps tak repeat karo
#   6. Optimizer step → LoRA weights update karo
#   7. Har eval_steps pe validation loss check karo
#   8. Har save_steps pe checkpoint save karo

train_result = trainer.train()
print("Training completed.")


In [ ]:
# Training log history print karo — har step ka loss dekho
# trainer.state.log_history → list of dicts, har step ki info
for log in trainer.state.log_history:
    print(log)


## 💾 Step 19: LoRA Adapter Save Karna

In [ ]:
# ============================================================
# 19. Trained Adapter aur Tokenizer Save Karna
# ============================================================
# Poora model nahi, sirf LoRA adapter save karte hain
# LoRA adapter = sirf woh thodi matrices jo hum ne train ki hain
# Size: usually 10-50 MB (TinyLlama base = ~2 GB)

# save_pretrained() → adapter weights + config files save karta hai
trainer.model.save_pretrained(config.adapter_dir)

# Tokenizer bhi saath save karo — inference ke waqt zaruri hoga
tokenizer.save_pretrained(config.adapter_dir)


In [ ]:
# Confirm karo ke files save ho gayi hain
print(f"LoRA adapter saved to: {config.adapter_dir}")
print("Saved files:")
print(os.listdir(config.adapter_dir))  # Kaunsi files save hui


## ☁️ Step 20: Hugging Face Hub Pe Upload (Optional)

In [ ]:
# ============================================================
# 20. LoRA Adapter Ko Hugging Face Hub Pe Push Karna
# ============================================================
# pehle huggingface-cli login karo ya HF_TOKEN environment variable set karo
# repo_id → "your_username/model_name" format mein apna username dalo

# repo_id = "iabubakar/pharma-tinyllama-domain-lora-live"


In [ ]:
# LoRA adapter Hub pe upload karo
# private=True → sirf tum dekh sako — publicly visible nahi
# model.push_to_hub(
#     repo_id,
#     private=True
# )


In [ ]:
# # Tokenizer bhi Hub pe upload karo — inference ke waqt zaruri hai
# tokenizer.push_to_hub(
#     repo_id,
#     private=True
# )


## 🔄 Step 21: Memory Free Karo — Inference Ke Liye Reload

In [ ]:
# ============================================================
# 21. Training Objects Delete Karo — Memory Free Karo
# ============================================================

# del → Python object delete karo — memory free hoti hai
del trainer

try:
    del model       # LoRA model delete karo
    del base_model  # Base model delete karo
except NameError:
    pass  # Agar pehle se delete ho chuke hain toh error nahi do

# Garbage collector chalao — unused Python objects finally free karo
gc.collect()

if use_cuda:
    # GPU ka cached memory bhi free karo
    torch.cuda.empty_cache()


In [ ]:
# Saved adapter folder se inference tokenizer load karo
from transformers import AutoTokenizer

# config.adapter_dir → wahan tokenizer bhi save kiya tha Step 19 mein
# use_fast=True → Rust-based fast tokenizer
inference_tokenizer = AutoTokenizer.from_pretrained(config.adapter_dir, use_fast=True)

if inference_tokenizer.pad_token is None:
    # EOS token ko pad token bana do (training jaise)
    inference_tokenizer.pad_token = inference_tokenizer.eos_token


In [ ]:
# Base model dobara load karo (GPU ya CPU pe)
if use_cuda:
    # 4-bit mein reload karo — same config jaise training mein thi
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    # CPU pe float32 mein load karo
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )


In [ ]:
# Saved LoRA adapter ko base model ke upar load karo
# PeftModel.from_pretrained() → base_model + saved adapter = fine-tuned model
from peft import PeftModel
inference_model = PeftModel.from_pretrained(inference_base_model, config.adapter_dir)


In [ ]:
# Model ko evaluation/inference mode mein set karo
# inference_model.eval() →
#   - Dropout layers off ho jaati hain (training mein on thi)
#   - BatchNorm fix ho jata hai
#   - Memory zyada efficient ho jati hai
inference_model.eval()


In [ ]:
# Confirm karo ke model inference ke liye ready hai
print("Base model + LoRA adapter loaded successfully for inference.")


## 🧠 Step 22: Text Generation Function

In [ ]:
# ============================================================
# 22. Text Continuation Generate Karne Ka Function
# ============================================================
# Yeh NON-INSTRUCTION model hai
# Ise Q&A style prompts nahi dene — seedha text ka shuru dena
# Model aage text continue karega

# prompt          → Text ka shuru (seed text)
# max_new_tokens  → Kitne naye tokens generate karo

def generate_completion(prompt: str, max_new_tokens: int = 120) -> str:

    # GPU ya CPU decide karo
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Prompt text ko token IDs mein convert karo
    # return_tensors="pt" → PyTorch tensors return karo (numpy nahi)
    # .to(device) → tensors ko GPU/CPU pe move karo
    inputs = inference_tokenizer(prompt, return_tensors="pt").to(device)

    # torch.no_grad() → gradient calculation band karo
    # Inference mein gradients ki zarurat nahi — memory aur speed dono improve
    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,                    # input_ids aur attention_mask unpack karo

            max_new_tokens=max_new_tokens, # Kitne naye tokens generate karo (120)

            do_sample=True,              # Random sampling use karo
                                         # False → hamesha most likely token (boring/repetitive)
                                         # True → thoda variety

            temperature=0.7,             # Creativity/randomness level
                                         # 0.0 = deterministic (same output hamesha)
                                         # 1.0 = fully random (chaotic)
                                         # 0.7 = thoda creative lekin mostly focused

            top_p=0.9,                   # Nucleus sampling
                                         # Top 90% probability tokens mein se choose karo
                                         # Bohot unlikely tokens filter ho jaate hain
                                         # Example: agar 100 tokens hain, cumulative prob 90% tak rakho

            repetition_penalty=1.1,      # Repeated words/phrases penalize karo
                                         # 1.0 = no penalty (same words baar baar aa sakte hain)
                                         # 1.1 = thoda penalty (repetition thodi kam hogi)
                                         # 2.0 = strict (repetition almost band)

            pad_token_id=inference_tokenizer.eos_token_id,  # Padding ke liye EOS use karo
        )

    # Generated token IDs → readable text
    # outputs[0] → pehla (aur akela) generated sequence
    # skip_special_tokens=True → [EOS], [PAD] jaise special tokens text mein mat dikhao
    return inference_tokenizer.decode(outputs[0], skip_special_tokens=True)


## 🧪 Step 23: Model Test Karna

In [ ]:
# ============================================================
# 23. Test Prompts Define Karo
# ============================================================
# Yeh continuation-style prompts hain — model inhe aage complete karega
# NON-INSTRUCTION model ke liye: "Q: ..." format NAHI use karna
# Seedha text ka shuru dena — jaise kisi article ka pehla sentence

prompts = [
    # Pharma domain — core topic
    "Metformin is one of the most widely prescribed oral antihyperglycemic agents",

    # Clinical trial style writing
    "Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe",

    # AI + Pharma cross-domain
    "Artificial intelligence is transforming pharmaceutical research by accelerating",
]


In [ ]:
# Har prompt ke liye text generate karo aur print karo
for prompt in prompts:
    print("=" * 100)
    print("PROMPT:")         # User ne kya diya
    print(prompt)
    print("\nMODEL CONTINUATION:")  # Model ne kya generate kiya
    # max_new_tokens=120 → 120 naye tokens aage generate karo
    print(generate_completion(prompt, max_new_tokens=120))
    print()


## 🔗 Step 24: Model Merge Karna — Deployment Ke Liye

In [ ]:
# ============================================================
# 24. LoRA Adapter Ko Base Model Mein Merge Karna
# ============================================================
# Normally model ke 2 alag hisse hain: Base Model + LoRA Adapter
# Merge ke baad: ek standalone complete model banta hai
# Kab use karo:
#   - vLLM, Ollama jaise production inference engines ke liye
#   - Mobile ya edge deployment ke liye
#   - Stage 2 (Instruction FT) mein merged model as base use karna hai

import os, torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

# Merged model save karne ka folder
merged_model_dir = "/content/pharma_tinyllama_merged_model"
os.makedirs(merged_model_dir, exist_ok=True)


In [ ]:
# Base model normal precision mein reload karo — safe merging ke liye
# dtype → float16 GPU pe, float32 CPU pe
base_model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",          # GPU/CPU automatically
    trust_remote_code=True,
)


In [ ]:
# Trained LoRA adapter base model ke upar load karo
# config.adapter_dir → wahan Stage 1 adapter save tha
model_with_adapter = PeftModel.from_pretrained(
    base_model,
    config.adapter_dir
)


In [ ]:
# LoRA weights base model mein permanently merge karo
# merge_and_unload():
#   - W_final = W_base + (alpha/r) × (A × B)  ← yeh calculation karta hai
#   - LoRA adapter hata deta hai
#   - Ek normal (non-PEFT) model return karta hai
merged_model = model_with_adapter.merge_and_unload()


In [ ]:
# Merged standalone model aur tokenizer save karo
merged_model.save_pretrained(merged_model_dir)
inference_tokenizer.save_pretrained(merged_model_dir)

print(f"Merged model saved to: {merged_model_dir}")
# Ab Stage 2 (Instruction FT) mein is merged model ko base ke tor pe load karenge


---
# 🟢 Stage 2: Instruction Fine-Tuning (IFT)
## Same Domain-Adapted Model Pe Q&A Sikhana

**Stage 1 mein kiya:** Raw pharma text se domain language seekhi (Non-Instruction FT)
**Stage 2 mein karenge:** Usi model pe instruction-response format sikhana

```
Base TinyLlama
    ↓
Stage 1: Raw pharma text → domain LoRA adapter
    ↓
Merged Stage 1 Model
    ↓
Stage 2: Pharma Q&A data → instruction LoRA adapter
    ↓
Instruction-tuned pharma model
```


In [ ]:
# Instruction dataset ka path set karo
# Yeh JSONL file honi chahiye jisme har line ek JSON object ho:
# {"instruction": "...", "input": "...", "output": "..."}
instruction_data_path = "/content/pharma_instruction_dataset.jsonl"


In [ ]:
# Datasets library se load_dataset import karo
# Yeh function JSON, CSV, JSONL, HuggingFace Hub — sab se data load kar sakta hai
from datasets import load_dataset


In [ ]:
# Instruction JSONL file load karo
# "json" → format batao (JSONL bhi "json" se hi load hota hai)
# data_files → file path
# split="train" → pura data ek hi split mein lo (baad mein khud split karenge)
instruction_dataset = load_dataset(
    "json",
    data_files=instruction_data_path,
    split="train"
)


In [ ]:
# Dataset ka overview print karo — kitne examples, kaunse columns
print(instruction_dataset)


In [ ]:
# Pehla raw instruction example dekho
# Fields honge: instruction, input, output
print(instruction_dataset[0])


## 📝 Instruction Records Ko Alpaca Format Mein Convert Karna

In [ ]:
# ============================================================
# Instruction Records Ko Training Text Mein Convert Karna
# ============================================================
# Alpaca format: Stanford Alpaca research paper ka training format
# Model is format se seekhta hai ke Q&A kaise karna hai

def format_instruction_record(record):
    # record se fields safely nikalo — .get() se KeyError nahi aata
    # .strip() → extra spaces hatao
    instruction = str(record.get("instruction", "")).strip()
    input_text = str(record.get("input", "")).strip()
    output_text = str(record.get("output", "")).strip()

    # Agar "input" field hai toh 3-part format use karo
    if input_text:
        text = (
            f"### Instruction:\n{instruction}\n\n"   # Instruction section
            f"### Input:\n{input_text}\n\n"          # Optional context/input
            f"### Response:\n{output_text}"            # Model ka jawab
        )
    else:
        # "input" nahi hai toh 2-part format use karo (zyada common)
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n{output_text}"
        )

    # Naya "text" field wapas karo — purani fields replace ho jaengi
    return {"text": text}


In [ ]:
# Har instruction record pe format function apply karo
# .map() → poori dataset pe ek function chalao
instruction_dataset = instruction_dataset.map(format_instruction_record)


In [ ]:
# Formatted dataset ka overview dekho — ab "text" column honi chahiye
instruction_dataset


In [ ]:
# Pehla formatted training example print karo — verify karo Alpaca format sahi hai
print(instruction_dataset[0]["text"])


## 🔀 Instruction Dataset Split

In [ ]:
# ============================================================
# Instruction Dataset Ko Train/Validation Mein Batna
# ============================================================

instruction_datasets = instruction_dataset.train_test_split(
    test_size=0.15,  # 15% validation
    seed=42          # Reproducible split
)

# "test" key ko "validation" naam dete hain — readable convention
instruction_datasets["validation"] = instruction_datasets.pop("test")

print(instruction_datasets)
print("Train examples:", len(instruction_datasets["train"]))
print("Validation examples:", len(instruction_datasets["validation"]))


## 🔤 Instruction Tokenizer Load Karna

In [ ]:
# ============================================================
# Instruction Fine-Tuning Ke Liye Tokenizer Load Karo
# ============================================================
# Base model ka tokenizer load karo — Stage 1 wala hi use hoga

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(tokenizer.pad_token)  # Confirm karo pad token set hua


In [ ]:
# Instruction examples ki maximum token length set karo
# 512 tokens = ek training example ki max length
instruction_max_length = 512


## 📦 Instruction Dataset Tokenize Karna (Padding + -100 Labels)

In [ ]:
# ============================================================
# Instruction Tokenization — Padding ke Saath aur -100 Labels
# ============================================================
# Stage 1 mein hum ne text packing use ki thi (bina padding ke)
# Stage 2 mein padding use karte hain — instruction data ka structure preserve hota hai
# Aur -100 labels use karte hain taake padding tokens loss mein count na hon

def tokenize_instruction_function(examples):
    # tokenizer() → text → token IDs
    # truncation=True     → 512 se zyada tokens? → cut kar do
    # padding="max_length"→ 512 se kam tokens? → padding tokens lagao
    # max_length=512      → har example exactly 512 tokens hoga
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )

    # Causal LM ke liye labels = input_ids
    # Model input[0:n-1] dekhta hai aur labels[1:n] predict karta hai
    tokens["labels"] = tokens["input_ids"].copy()

    # ── -100 Labels: Padding Ignore Karo ─────────────────────────────────────
    # attention_mask = 1 → real token, 0 → padding token
    # Padding tokens ke liye label = -100 set karo
    # PyTorch CrossEntropyLoss -100 ko automatically ignore karta hai
    # Matlab: model sirf real instruction text se seekhega, padding se nahi
    tokens["labels"] = [
        [
            # Real token → original token ID rakho (model is se seekhe)
            # Padding token → -100 rakho (model is se nahi seekhega)
            token if mask == 1 else -100
            for token, mask in zip(input_ids, attention_mask)
        ]
        for input_ids, attention_mask in zip(tokens["input_ids"], tokens["attention_mask"])
    ]

    return tokens


In [ ]:
# Instruction dataset tokenize karo
# batched=True → ek baar mein kai examples process karo (fast)
# remove_columns → original "text", "instruction" etc. columns hata do
instruction_tokenized_datasets = instruction_datasets.map(
    tokenize_instruction_function,
    batched=True,
    remove_columns=instruction_datasets["train"].column_names,
    desc="Tokenizing instruction dataset",
)

print(instruction_tokenized_datasets)


## 📌 Note: Do Approaches — Stage 1 Adapter Continue Karo Ya Merged Model Use Karo

| | Approach 1: Same Stage 1 LoRA Continue | Approach 2: Merge Then New LoRA |
|---|---|---|
| **Flow** | Stage 1 adapter continue train karo | Stage 1 merge karo, naya LoRA lagao |
| **Complexity** | Simple | Complex (merge step zaroori) |
| **Is notebook mein** | ❌ (commented out) | ✅ Yeh use ho raha hai |
| **Why** | Simpler hai lekin Stage 1 domain knowledge thodi kam effective | Merged model = Stage 1 fully baked in |


In [ ]:
# ── Approach 1 (Commented Out): Stage 1 LoRA Adapter Continue Karna ────────────
# Yeh approach simpler hai — same adapter aage train karo
# Lekin is notebook mein Approach 2 use ho raha hai (merged model)

# gc.collect()
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()
# use_cuda = torch.cuda.is_available()
# if use_cuda:
#     instruction_base_model = AutoModelForCausalLM.from_pretrained(
#         config.model_name,
#         quantization_config=BitsAndBytesConfig(...),
#         ...
#     )
#     instruction_base_model = prepare_model_for_kbit_training(instruction_base_model)
# else:
#     instruction_base_model = AutoModelForCausalLM.from_pretrained(...)
# instruction_base_model.config.use_cache = False

# # Stage 1 adapter load karo aur trainable rakho
# instruction_model = PeftModel.from_pretrained(
#     instruction_base_model,
#     config.adapter_dir,
#     is_trainable=True,  # ← yeh important hai — adapter train hoga
# )
# instruction_model.print_trainable_parameters()


## 🤖 Merged Stage 1 Model Pe Naya LoRA Adapter — Instruction FT

In [ ]:
# ============================================================
# Merged Stage 1 Model Load Karo + Naya LoRA Adapter Lagao
# ============================================================
# Approach 2: merged_model_dir se base load karo, naya LoRA add karo

import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Memory free karo
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

use_cuda = torch.cuda.is_available()

# Stage 1 ka merged model kahan save tha
merged_model_dir = "/content/pharma_tinyllama_merged_model"

if use_cuda:
    # Merged Stage 1 model 4-bit mein load karo — GPU efficient
    instruction_base_model = AutoModelForCausalLM.from_pretrained(
        merged_model_dir,                # Stage 1 merged model yahan tha
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,           # 4-bit quantization — kam memory
            bnb_4bit_quant_type="nf4",   # NormalFloat4 — best quality
            bnb_4bit_compute_dtype=torch.float16,  # Fast computation
            bnb_4bit_use_double_quant=True,         # Extra compression
        ),
        device_map="auto",
        trust_remote_code=True,
    )
    instruction_base_model = prepare_model_for_kbit_training(instruction_base_model)

else:
    # CPU fallback — training slow hogi
    instruction_base_model = AutoModelForCausalLM.from_pretrained(
        merged_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

# Training mein cache disable karo
instruction_base_model.config.use_cache = False

# ── Naya LoRA Adapter Instruction FT Ke Liye ─────────────────────────────────
# Yeh adapter Stage 2 instruction following seekhega
# Same LoRA config jaise Stage 1 mein tha
instruction_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,  # Causal LM task
    r=16,                          # Rank 16 — balanced quality aur efficiency
    lora_alpha=32,                 # Scaling factor
    lora_dropout=0.05,             # 5% dropout — overfitting roko
    bias="none",                   # Bias train nahi karna
    target_modules=[               # Attention aur FFN layers
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

# Merged base model pe naya LoRA inject karo
instruction_model = get_peft_model(
    instruction_base_model,
    instruction_lora_config
)

# Kitne parameters trainable hain — dekho
instruction_model.print_trainable_parameters()


In [ ]:
# ============================================================
# Instruction FT Data Collator
# ============================================================
# Stage 2 ka data collator — same mlm=False — causal LM hai

from transformers import DataCollatorForLanguageModeling
instruction_data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal LM — next token prediction
)


In [ ]:
# ============================================================
# Instruction FT Ke Liye Output Folders Banao
# ============================================================

instruction_output_dir = "/content/pharma_tinyllama_instruction_lora_output"
instruction_adapter_dir = "/content/pharma_tinyllama_instruction_lora_adapter"

# exist_ok=True → agar pehle se hain toh error nahi
os.makedirs(instruction_output_dir, exist_ok=True)
os.makedirs(instruction_adapter_dir, exist_ok=True)


In [ ]:
# ============================================================
# Instruction Fine-Tuning Training Arguments
# ============================================================
from transformers import TrainingArguments

instruction_training_args = TrainingArguments(
    output_dir=instruction_output_dir,  # Checkpoints yahan save honge

    # 5 epochs — instruction data chota hota hai, 5 baar dikhana kaafi
    num_train_epochs=5,
    max_steps=-1,                   # Epochs se control karo

    # Batch settings — Stage 1 se same
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,  # Effective batch = 8

    # Optimizer — Stage 1 se thoda kam learning rate
    # Instruction data zyada sensitive hota hai — careful learning chahiye
    learning_rate=1e-4,             # 0.0001 — Stage 1 (2e-4) se half
    warmup_steps=5,                 # Pehle 5 steps slowly barhao LR
    weight_decay=0.01,              # L2 regularization

    # Logging — har step print karo (small dataset ke liye)
    logging_steps=1,
    logging_first_step=True,

    # Har step pe validation chalao — instruction quality monitor karo
    eval_strategy="steps",
    eval_steps=1,

    # Checkpoints
    save_steps=25,
    save_total_limit=2,

    # Precision
    fp16=use_cuda,   # GPU ho toh float16
    bf16=False,

    report_to="none",             # External logging band
    remove_unused_columns=False,
)

print(instruction_training_args)


In [ ]:
# ============================================================
# Instruction Trainer Build Karo
# ============================================================
from transformers import Trainer

instruction_trainer = Trainer(
    model=instruction_model,                            # Stage 2 LoRA model
    args=instruction_training_args,                     # Upar wali settings
    train_dataset=instruction_tokenized_datasets["train"],      # Q&A training data
    eval_dataset=instruction_tokenized_datasets["validation"],  # Q&A validation data
    data_collator=instruction_data_collator,            # Batch bananewala
)

print("Instruction Trainer is ready.")


In [ ]:
# ============================================================
# Instruction Fine-Tuning Shuru Karo!
# ============================================================
# Ab model sikhega:
# "### Instruction: Explain metformin" → "### Response: Metformin works by..."
# Yeh Q&A style follow karna seekhega

instruction_train_result = instruction_trainer.train()


In [ ]:
# Training result print karo
print("Instruction fine-tuning completed.")
print(instruction_train_result)


In [ ]:
# ============================================================
# Final Instruction-Tuned LoRA Adapter Save Karna
# ============================================================
# Yeh adapter mein hai:
# Stage 1 domain knowledge (pharma language) + Stage 2 instruction following (Q&A)

import os

instruction_adapter_dir = "/content/pharma_tinyllama_instruction_lora_adapter"
os.makedirs(instruction_adapter_dir, exist_ok=True)

# Adapter weights save karo
instruction_trainer.model.save_pretrained(instruction_adapter_dir)
# Tokenizer bhi save karo — inference ke liye zaruri
tokenizer.save_pretrained(instruction_adapter_dir)

print(f"Final instruction-tuned LoRA adapter saved to: {instruction_adapter_dir}")
print(os.listdir(instruction_adapter_dir))


In [ ]:
# ============================================================
# Instruction Model Reload Karo — Inference Ke Liye
# ============================================================

# Memory free karo
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if use_cuda:
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

# Instruction adapter base model pe load karo
final_instruction_model = PeftModel.from_pretrained(
    base_model,
    instruction_adapter_dir,  # Stage 2 adapter
)

# Inference mode on karo
final_instruction_model.eval()

print("Final instruction-tuned model loaded successfully.")


In [ ]:
# ============================================================
# Instruction-Style Inference Functions
# ============================================================

def build_instruction_prompt(instruction, input_text=""):
    # Alpaca format mein prompt banao
    instruction = instruction.strip()
    input_text = input_text.strip()

    if input_text:
        # Input field hai toh 3-part format
        return (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n"  # Yahan model continue karega
        )

    # Input nahi hai toh 2-part format
    return (
        f"### Instruction:\n{instruction}\n\n"
        f"### Response:\n"
    )


def generate_instruction_response(instruction, input_text="", max_new_tokens=150):
    # Step 1: Prompt banao
    prompt = build_instruction_prompt(instruction, input_text)

    # Step 2: Tokenize karo aur model ke device pe bhejo
    inputs = tokenizer(
        prompt,
        return_tensors="pt"  # PyTorch tensors
    ).to(final_instruction_model.device)

    # Step 3: Generate karo (gradients band)
    with torch.no_grad():
        outputs = final_instruction_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,  # 150 naye tokens

            do_sample=True,       # Random sampling
            temperature=0.7,      # Creativity level (0.7 = balanced)
            top_p=0.9,            # Nucleus sampling — top 90% tokens
            repetition_penalty=1.1, # Repetition kam karo

            pad_token_id=tokenizer.eos_token_id,  # Padding ke liye EOS
            eos_token_id=tokenizer.eos_token_id,  # Yahan generation roko
        )

    # Step 4: Token IDs → readable text
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
# ============================================================
# Instruction-Tuned Model Test Karo
# ============================================================
# Ab Q&A style prompts de sakte hain — model properly jawab dega

test_questions = [
    # Pharma mechanism question
    "Explain the primary mechanism of action of metformin.",

    # Drug combination question
    "Why can atorvastatin and ezetimibe reduce LDL-C more effectively together?",

    # Drug delivery question
    "Summarize the role of lipid nanoparticles in mRNA vaccines.",

    # AI in pharma question
    "Why should AI predictions in drug discovery be experimentally validated?",
]

for question in test_questions:
    print("=" * 100)
    print("QUESTION:")      # User ka Q&A style sawaal
    print(question)
    print("\nMODEL RESPONSE:")  # Model ka detailed jawab
    print(generate_instruction_response(question, max_new_tokens=150))


## 🔗 Instruction Model Merge Karna — Stage 3 Ke Liye Base

In [ ]:
# ============================================================
# Instruction-Tuned LoRA Adapter Ko Base Model Mein Merge Karna
# ============================================================
# Merged model Stage 3 (Preference Tuning) ka base ban jayega
# Yeh important hai: preference tuning merged model pe kaam karta hai

import os, gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Merged instruction model save folder
merged_instruction_model_dir = "/content/pharma_tinyllama_instruction_merged_model"
os.makedirs(merged_instruction_model_dir, exist_ok=True)

# Original base model normal precision mein load karo (merge ke liye float16 safe hai)
base_model_for_merge = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

# Tokenizer load karo
tokenizer_for_merge = AutoTokenizer.from_pretrained(
    config.model_name,
    trust_remote_code=True,
)
if tokenizer_for_merge.pad_token is None:
    tokenizer_for_merge.pad_token = tokenizer_for_merge.eos_token

# Instruction LoRA adapter base model pe lagao
model_with_instruction_adapter = PeftModel.from_pretrained(
    base_model_for_merge,
    instruction_adapter_dir,  # Stage 2 instruction adapter
)

# Adapter weights base model mein permanently merge karo
# W_final = W_base + (32/16) × (A × B)
merged_instruction_model = model_with_instruction_adapter.merge_and_unload()

# Merged model save karo — Stage 3 is folder ko use karega
merged_instruction_model.save_pretrained(merged_instruction_model_dir)
tokenizer_for_merge.save_pretrained(merged_instruction_model_dir)

print(f"Merged instruction-tuned model saved to: {merged_instruction_model_dir}")


# Stage 3: Preference Tuning with DPO

In Stage 1, we adapted the model to the pharma domain using raw non-instruction text.

In Stage 2, we instruction-tuned the model using instruction-response data.

In Stage 3, we will use **preference data** with DPO.

DPO data has three main columns:

```text
prompt
chosen
rejected
```

- `prompt` is the user instruction.
- `chosen` is the preferred/better answer.
- `rejected` is the weaker answer.

The goal of DPO is to make the model prefer the `chosen` response over the `rejected` response.

Paper link: https://arxiv.org/pdf/2305.18290

| Section                       | Simple Meaning                                                                                         | Key Point                                                                           |
| ----------------------------- | ------------------------------------------------------------------------------------------------------ | ----------------------------------------------------------------------------------- |
| **DPO Full Form**             | Direct Preference Optimization                                                                         | The model is trained directly using preference data.                                |
| **Paper**                     | “Direct Preference Optimization”                                                                       | Published at NeurIPS 2023 by Stanford researchers.                                  |
| **Main Idea**                 | Teach the model which answer is better and which answer is weaker.                                     | The model learns to prefer the `chosen` answer and avoid the `rejected` answer.     |
| **Before DPO: RLHF**          | RLHF usually has three stages.                                                                         | SFT → Reward Model → PPO                                                            |
| **RLHF Problem**              | RLHF is complex, expensive, and unstable.                                                              | Training a reward model and using PPO require high compute and careful tuning.      |
| **DPO Insight**               | A separate reward model is not required.                                                               | The language model itself can act like an implicit reward model.                    |
| **DPO Dataset Format**        | Each sample has three main fields.                                                                     | `prompt`, `chosen`, and `rejected`                                                  |
| **Prompt**                    | The user question or instruction.                                                                      | Example: “Explain the mechanism of metformin.”                                      |
| **Chosen**                    | The better or preferred answer.                                                                        | Usually accurate, complete, safe, and well-structured.                              |
| **Rejected**                  | The weaker or rejected answer.                                                                         | Usually vague, incomplete, incorrect, or unsafe.                                    |
| **DPO Training Goal**         | Increase the probability of the preferred answer.                                                      | The model becomes more likely to generate answers like the `chosen` response.       |
| **Role of Rejected Answer**   | Shows the model what type of answer to avoid.                                                          | The model reduces the probability of the `rejected` style answer.                   |
| **Reference Model**           | Usually the SFT model.                                                                                 | It prevents the DPO model from drifting too far from the original fine-tuned model. |
| **Policy Model**              | The model being trained during DPO.                                                                    | It learns to prefer the `chosen` answer over the `rejected` answer.                 |
| **Beta β**                    | A control parameter.                                                                                   | It controls how strongly the model moves away from the reference model.             |
| **DPO Loss**                  | A binary classification-style loss.                                                                    | It trains the model to make the `chosen` answer win over the `rejected` answer.     |
| **Reward Model Needed?**      | No.                                                                                                    | DPO removes the need for separate reward model training.                            |
| **PPO Needed?**               | No.                                                                                                    | DPO works more like supervised training instead of reinforcement learning.          |
| **Sampling During Training?** | No.                                                                                                    | DPO does not require an expensive generation loop like PPO.                         |
| **Main Advantage**            | Simpler and more stable.                                                                               | Easier to implement compared to traditional RLHF.                                   |
| **Training Cost**             | Lower than RLHF.                                                                                       | Only the policy model is trained.                                                   |
| **DPO vs SFT**                | SFT teaches the model how to answer.                                                                   | DPO teaches the model which answer is better.                                       |
| **DPO vs RLHF**               | RLHF uses a reward model and PPO.                                                                      | DPO directly uses preference loss.                                                  |
| **Gradient Intuition**        | Stronger updates happen when the model ranks answers incorrectly.                                      | The model learns more from difficult examples.                                      |
| **Practical Pipeline**        | Start with an SFT model, add preference data, then train with DPO.                                     | This creates a simple alignment pipeline.                                           |
| **Common Beta Value**         | The paper commonly used `β = 0.1`.                                                                     | For summarization tasks, `β = 0.5` was also used.                                   |
| **Experiments**               | Tested on sentiment, summarization, and dialogue tasks.                                                | DPO can perform equal to or better than PPO.                                        |
| **Limitation**                | Large-scale training, reward hacking, and out-of-distribution generalization are still open questions. | DPO is powerful, but not perfect.                                                   |
| **Classroom One-Liner**       | DPO teaches the model which answer is better.                                                          | Instruction tuning teaches answering; DPO teaches preference.                       |


In [ ]:
# ============================================================
# 25. Install TRL for DPO training
# ============================================================
# TRL provides DPOTrainer and DPOConfig for preference tuning.

!pip install -q -U trl

In [ ]:
# ============================================================
# 26. Load DPO preference dataset
# ============================================================
# Expected columns: prompt, chosen, rejected

from datasets import load_dataset

preference_data_path = "/content/pharma_preference_dataset.jsonl"

preference_dataset = load_dataset(
    "json",
    data_files=preference_data_path,
    split="train"
)

print(preference_dataset)
print(preference_dataset[0])


In [ ]:
# Create train-validation split
preference_dataset = preference_dataset.train_test_split(
    test_size=0.15,
    seed=42
)

# Rename test split to validation split
preference_dataset["validation"] = preference_dataset.pop("test")

print("After train-validation split:")
print(preference_dataset)
print("Train rows:", len(preference_dataset["train"]))
print("Validation rows:", len(preference_dataset["validation"]))

## Preference Tuning Base Model

For DPO, we use the **merged instruction-tuned model** as the base model.

Then we attach a **new LoRA adapter** for preference tuning.

This gives us the flow:

```text
Merged instruction-tuned model
        +
New preference LoRA adapter
        ↓
DPO preference tuning
```


In [ ]:
merged_instruction_model_dir


In [ ]:
# ============================================================
# Load merged instruction model as base for preference tuning
# ============================================================

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

use_cuda = torch.cuda.is_available()

if use_cuda:
    preference_base_model = AutoModelForCausalLM.from_pretrained(
        merged_instruction_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )

    preference_base_model = prepare_model_for_kbit_training(preference_base_model)

else:
    preference_base_model = AutoModelForCausalLM.from_pretrained(
        merged_instruction_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

preference_base_model.config.use_cache = False

# Create a new LoRA adapter for preference tuning.
preference_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

preference_model = get_peft_model(
    preference_base_model,
    preference_lora_config,
)

preference_model.print_trainable_parameters()

In [ ]:
# ============================================================
# 30. Configure DPO training
# ============================================================

import os
#import inspect
#from transformers import TrainingArguments
from trl import DPOTrainer
from trl import DPOConfig

try:
    from trl import DPOConfig
    has_dpo_config = True
except ImportError:
    DPOConfig = None
    has_dpo_config = False

preference_output_dir = "/content/pharma_tinyllama_preference_dpo_output"
preference_adapter_dir = "/content/pharma_tinyllama_preference_dpo_lora_adapter"

os.makedirs(preference_output_dir, exist_ok=True)
os.makedirs(preference_adapter_dir, exist_ok=True)

In [ ]:
# ============================================================
# 30. Create DPO training arguments - Simple Version
# ============================================================

from trl import DPOConfig

dpo_training_args = DPOConfig(
    output_dir=preference_output_dir,

    # Training duration
    num_train_epochs=3,
    max_steps=5,

    # Batch settings
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    # Optimizer settings
    learning_rate=5e-5,
    warmup_steps=2,
    weight_decay=0.01,

    # Logging and evaluation
    logging_steps=1,
    logging_first_step=True,
    eval_strategy="steps",
    eval_steps=1,

    # Checkpoint saving
    save_steps=5,
    save_total_limit=2,

    # Precision settings
    fp16=False,
    bf16=False,

    # Disable external logging tools
    report_to="none",

    # Keep required columns
    remove_unused_columns=False,

    # DPO hyperparameter
    beta=0.1,
)

print(dpo_training_args)

In [ ]:
# ============================================================
# 31. Build DPOTrainer - Simple Student Version
# ============================================================

from trl import DPOTrainer

dpo_trainer = DPOTrainer(
    model=preference_model,
    ref_model=None,  # None means TRL will internally use the reference behavior
    args=dpo_training_args,

    train_dataset=preference_dataset["train"],
    eval_dataset=preference_dataset["validation"],

    processing_class=tokenizer,
)

print("DPOTrainer is ready.")

In [ ]:
# ============================================================
# 32. Start DPO preference tuning
# ============================================================

dpo_train_result = dpo_trainer.train()

print("DPO preference tuning completed.")
print(dpo_train_result)


| Parameter               | Short Meaning                                                                              |
| ----------------------- | ------------------------------------------------------------------------------------------ |
| **Step**                | Current optimizer step during training.                                                    |
| **Training Loss**       | DPO loss on the training data; lower is generally better.                                  |
| **Validation Loss**     | DPO loss on unseen validation data; helps check generalization.                            |
| **Entropy**             | Measures how uncertain the model is; higher means more random, lower means more confident. |
| **Num Tokens**          | Total number of tokens processed so far.                                                   |
| **Logits/chosen**       | Raw model score for the preferred answer.                                                  |
| **Logits/rejected**     | Raw model score for the rejected answer.                                                   |
| **Mean Token Accuracy** | Average token-level prediction accuracy.                                                   |
| **Rewards/chosen**      | DPO implicit reward for the preferred answer; should be higher.                            |
| **Rewards/rejected**    | DPO implicit reward for the rejected answer; should be lower.                              |
| **Rewards/accuracies**  | How often the model ranks the chosen answer above the rejected answer.                     |
| **Rewards/margins**     | Difference between chosen reward and rejected reward; positive is good.                    |
| **Logps/chosen**        | Log probability of the chosen answer; less negative means more likely.                     |
| **Logps/rejected**      | Log probability of the rejected answer; ideally more negative than chosen.                 |


Simple summary: In DPO training, the main goal is to make the model assign higher probability and higher reward to the chosen answer than the rejected answer.

In [ ]:
# ============================================================
# 33. Save DPO preference-tuned LoRA adapter
# ============================================================

dpo_trainer.model.save_pretrained(preference_adapter_dir)
tokenizer.save_pretrained(preference_adapter_dir)

print(f"Preference-tuned LoRA adapter saved to: {preference_adapter_dir}")
print(os.listdir(preference_adapter_dir))


In [ ]:
# # ============================================================
# # Push Stage 3 DPO LoRA adapter to Hugging Face
# # ============================================================

# dpo_trainer.model.push_to_hub(
#     HF_REPO_DPO_ADAPTER,
#     private=True
# )

# tokenizer.push_to_hub(
#     HF_REPO_DPO_ADAPTER,
#     private=True
# )

# print("Stage 3 DPO LoRA adapter pushed to:")
# print(HF_REPO_DPO_ADAPTER)

In [ ]:
# ============================================================
# 34. Reload preference-tuned model for inference
# ============================================================

import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

if use_cuda:
    preference_inference_base_model = AutoModelForCausalLM.from_pretrained(
        merged_instruction_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    preference_inference_base_model = AutoModelForCausalLM.from_pretrained(
        merged_instruction_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

preference_inference_model = PeftModel.from_pretrained(
    preference_inference_base_model,
    preference_adapter_dir,
)

preference_inference_model.eval()

print("Preference-tuned model loaded successfully for inference.")


In [ ]:
# ============================================================
# 35. Preference-tuned inference helper
# ============================================================

def build_preference_prompt(instruction, input_text=""):
    instruction = instruction.strip()
    input_text = input_text.strip()

    if input_text:
        return (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n"
        )

    return (
        f"### Instruction:\n{instruction}\n\n"
        f"### Response:\n"
    )

In [ ]:
def generate_preference_response(instruction, input_text="", max_new_tokens=150):
    prompt = build_preference_prompt(instruction, input_text)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(preference_inference_model.device)

    with torch.no_grad():
        outputs = preference_inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
# ============================================================
# 36. Test preference-tuned pharma model
# ============================================================

preference_test_questions = [
    "Explain the primary mechanism of action of metformin.",
    "Why should AI predictions in drug discovery be experimentally validated?",
    "Define pharmacovigilance.",
    "Explain why pharmacovigilance continues after drug approval.",
]

for question in preference_test_questions:
    print("=" * 100)
    print("QUESTION:")
    print(question)

    print("\nMODEL RESPONSE:")
    print(generate_preference_response(question, max_new_tokens=150))


In [ ]:
# ============================================================
# 37. Optional: Merge DPO preference adapter into the instruction-tuned base model
# ============================================================
# Use this only after preference tuning is complete and you want a standalone final model.

import os
import gc
import torch

from transformers import AutoModelForCausalLM
from peft import PeftModel

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

final_merged_preference_model_dir = "/content/pharma_tinyllama_final_preference_merged_model"
os.makedirs(final_merged_preference_model_dir, exist_ok=True)

In [ ]:
# Load the merged instruction model in normal precision for safe merging.
base_model_for_preference_merge = AutoModelForCausalLM.from_pretrained(
    merged_instruction_model_dir,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

# Attach the DPO preference LoRA adapter.
model_with_preference_adapter = PeftModel.from_pretrained(
    base_model_for_preference_merge,
    preference_adapter_dir,
)

# Merge the preference adapter into the instruction-tuned base model.
final_merged_preference_model = model_with_preference_adapter.merge_and_unload()

# Save final standalone model and tokenizer.
final_merged_preference_model.save_pretrained(final_merged_preference_model_dir)
tokenizer.save_pretrained(final_merged_preference_model_dir)

print(f"Final merged preference-tuned model saved to: {final_merged_preference_model_dir}")

---
## ✅ Poora Pipeline Complete!

**Stage 1 (Non-Instruction FT):**
PDF → Clean → Paragraphs → 512-token blocks → TinyLlama + LoRA → Domain text continuation

**Stage 2 (Instruction FT):**
Merged Stage 1 → Q&A Alpaca format data → -100 labels → Instruction following Q&A

**Stage 3 (Preference FT) — Base Ready:**
Merged Stage 2 → Preference LoRA ready → Agle notebook mein chosen/rejected pairs se train hoga

> *"Teen stages mein poora LLM fine-tuning pipeline — industrial grade!"*
